# Little Sree — Saree Kada site assistant

This Gradio notebook runs the same site-scoped chat agent as the Saree Kada website. It answers only from the site's product catalog and published information.

Install the notebook dependencies in the next cell. Set `GOOGLE_API_KEY` in a local `.env` file at the repository root before starting the chat. Never put an API key in a notebook cell or commit it.

In [ ]:
%pip install openai python-dotenv gradio

In [ ]:
import json
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=False)

catalog_path = Path('public/products.json')
if not catalog_path.exists():
    raise FileNotFoundError('Run this notebook from the Saree Kada repository root.')

products = json.loads(catalog_path.read_text(encoding='utf-8'))
product_facts = '\n'.join(
    f"- {item['name']}: {item['price']}; color {item['color']}; "
    f"composition {item['composition']}; woven in {item['origin']}."
    for item in products
)

site_context = f'''
Saree Kada is an independent Kerala saree storefront rooted in Kochi. Its point of view is handwoven clothing designed for movement and made to last. The site discusses Kerala craft, kasavu borders, Theyyam colour, natural fibers, small weaving communities, and everyday ceremony.

Catalog (prices are in Indian rupees):
{product_facts}

Other published details: free delivery across India on orders over ₹5,000; product pages state that a blouse piece is included and recommend cold washing and hanging to dry. The site displays Instagram handle @auraform.studio. Do not invent stock availability, delivery dates, return policies, contact details, sizing guarantees, or other product facts.

Regional and cultural background (general context, not verified provenance for each individual product):
- Kasavu Saree (listed origin: Balaramapuram, Kerala): Kasavu refers to the gold-coloured border associated with Kerala's traditional off-white cotton clothing. Kasavu garments are strongly associated with festive and ceremonial dress, including Onam and Vishu. Balaramapuram, in Thiruvananthapuram district, is a long-established handloom centre known for fine cotton cloth and kasavu borders. The catalog does not identify this saree's weaver, production date, or the precise history of its border.
- Handwoven White Set Saree (listed origin: Kuthampully, Kerala): Kuthampully, in Thrissur district, is a recognized handloom village with a cotton-weaving tradition carried by generations of weaving families. Local accounts commonly connect the settlement's weaving community with migration from Karnataka and patronage by the former Cochin royal household; details and dates vary between accounts. The catalog confirms the listed origin and handwoven Kerala cotton composition, but not an individual weaver or a specific family lineage.
- Theyyam Silk Saree (listed origin: Kannur, Kerala): Theyyam is a ritual performance tradition of North Kerala, especially associated with Kannur and Kasaragod, with vivid colours, elaborate face painting, and ceremonial costume. Its visual language can inspire contemporary textiles, but this product is listed as silk cotton with a hand-finished border; the catalog does not claim it is a traditional Theyyam costume or that its cloth was used in a ritual.

When asked for history, share this relevant general background and clearly distinguish it from what the catalog verifies. Explain that oral traditions may vary. Do not invent dates, artisan identities, royal commissions, or a direct historical link between a product and a ritual. If asked for an item's specific maker, exact age, or documented provenance, say the site does not provide it.
'''

system_prompt = f'''
You are Little Sree, Saree Kada's friendly AI chat guide. You answer questions about Saree Kada, its catalog, and relevant Kerala textile and cultural history. Use catalog facts for claims about specific products and use the clearly labeled regional background for general history. Explain when something is general cultural context rather than documented provenance of an individual piece. If asked about something unrelated to Saree Kada or Kerala textile context, politely steer back to the site. Do not invent product-specific makers, dates, family histories, royal commissions, stock, delivery promises, or policies. Qualify oral traditions where accounts vary. If asked about a product fact not provided, say the site does not specify it. If asked, clearly identify yourself as an AI assistant, not a human. Be warm, concise, and useful.

SITE CONTEXT:
{site_context}
'''

api_key = os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY')
if not api_key:
    raise RuntimeError('Set GOOGLE_API_KEY in a local .env file before running this cell.')

client = OpenAI(
    base_url='https://generativelanguage.googleapis.com/v1beta/openai/',
    api_key=api_key,
)
model = os.getenv('GOOGLE_MODEL', 'gemini-3.5-flash-lite')
print(f'Little Sree is ready to answer questions about {len(products)} catalog items.')

In [ ]:
def chat(message, history):
    safe_history = [
        {'role': item['role'], 'content': item['content']}
        for item in history[-10:]
        if item.get('role') in ('user', 'assistant') and isinstance(item.get('content'), str)
    ]
    messages = [
        {'role': 'system', 'content': system_prompt},
        *safe_history,
        {'role': 'user', 'content': message[:2000]},
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.4,
        max_tokens=500,
    )
    return response.choices[0].message.content or 'I do not have an answer for that from the Saree Kada site.'

demo = gr.ChatInterface(
    fn=chat,
    type='messages',
    title='Little Sree',
    description='Ask me about Saree Kada, its sarees, and what is published on this site.',
    examples=[
        'Which sarees are in the collection?',
        'What is the Kasavu Saree made from?',
        'How much is delivery?',
    ],
)
demo.launch()